# 🎨 [Day 47_2] 하이브리드 검색 & 질의 변환 — 핸즈온 실습

> API 키, DB 연결 **전혀 필요 없습니다.** 아래 셀을 순서대로 실행만 하면
> BM25 + 벡터검색 + RRF + 질의변환을 눈으로 직접 확인할 수 있습니다.

## 🗺️ 전체 지도에서 지금 위치
```text
[Day 46] 청킹 전략 & RAPTOR .................... 완료 ✅
[Day 47] 하이브리드 검색 & 질의 변환 ............ ◀◀◀ 지금 여기
[Day 48] Cross-Encoder 리랭킹 ................... 다음
```

## 💡 10초 비유
- **BM25(사서)**: 색인카드에서 "정정요청"이란 글자를 정확히 찾아줌. 표현이 다르면 못 찾음.
- **벡터검색(독심술사)**: 뜻이 통하면 표현이 달라도 찾아줌. 짧고 딱딱한 문장은 오히려 헷갈려함.
- **RRF(공정한 심판)**: 둘의 순위표를 모아서 "양쪽에서 다 인정받은 문서"를 1등으로 뽑음.

## 1. 실습용 가짜 문서 준비 (우리 미술입시 도메인 느낌, 실제 DB 아님)

In [ ]:
docs = [
    {"id": "D1", "text": "미술활동보고서는 600자 이내로 작성해야 하며 활동의 내용과 성과를 구체적으로 기술한다"},
    {"id": "D2", "text": "정정요청은 평가자가 지원자에게 전달할 사유를 100자 이내로 작성하고 정정요청 버튼을 누른다"},
    {"id": "D3", "text": "평가거부는 간단한 사유를 100자 이내로 작성한 후 평가거부 버튼을 누르면 완료된다"},
    {"id": "D4", "text": "수채화 물감과 아크릴 물감 파스텔 색연필 연필 등을 준비물로 지참한다"},
    {"id": "D5", "text": "논술고사 시험시간은 120분이며 인문계열과 예술학과가 함께 응시한다"},
    {"id": "D6", "text": "평가자 회원가입은 행정전자서명 인증서로 로그인한 뒤 재직정보를 등록한다"},
    {"id": "D7", "text": "재직사실확인서는 다운로드 받아 작성한 후 팩스로 입학관리본부에 전송한다"},
]
print(f"문서 {len(docs)}개 준비 완료")

## 2. [검색기 1] BM25 — 사서 방식 (외부 패키지 불필요, 순수 파이썬)

In [ ]:
import re, math
from collections import Counter

def tokenize(text):
    words = re.sub(r"[^가-힣a-zA-Z0-9\s]", " ", text).split()
    toks = []
    for w in words:
        toks.append(w)
        for i in range(len(w) - 1):
            toks.append(w[i:i+2])  # 바이그램 - 조사가 붙어도 부분매칭되게
    return toks

class BM25:
    def __init__(self, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
    def fit(self, docs):
        self.docs = docs
        self.tf = [Counter(tokenize(d["text"])) for d in docs]
        self.dl = [sum(c.values()) for c in self.tf]
        self.avgdl = sum(self.dl) / len(self.dl)
        self.df = Counter()
        for c in self.tf:
            for t in c: self.df[t] += 1
        self.N = len(docs)
    def search(self, query, top_k=5):
        q = tokenize(query)
        scores = []
        for i, doc in enumerate(self.docs):
            s = 0.0
            for t in q:
                if t not in self.tf[i]: continue
                idf = math.log((self.N - self.df[t] + 0.5) / (self.df[t] + 0.5) + 1)
                tf = self.tf[i][t]
                s += idf * tf * (self.k1 + 1) / (tf + self.k1 * (1 - self.b + self.b * self.dl[i] / self.avgdl))
            scores.append((doc["id"], s))
        scores.sort(key=lambda x: -x[1])
        return [{"rank": r+1, "doc_id": d, "score": round(s,3)} for r,(d,s) in enumerate(scores[:top_k])]

bm25 = BM25()
bm25.fit(docs)
print("질의: '정정요청 방법'")
for r in bm25.search("정정요청 방법"):
    print(r)

## 3. [검색기 2] 벡터검색 — 독심술사 방식 (가짜지만 결정적인 임베딩, API 키 불필요)

In [ ]:
import hashlib

def stable_hash(word):
    """파이썬 내장 hash()는 실행할 때마다 값이 달라져서(보안상 랜덤 시드) 노트북을
    다시 실행하면 결과가 바뀝니다 - 교재용 코드는 항상 같은 결과가 나와야 하므로
    hashlib로 고정된 값을 씁니다."""
    return int(hashlib.md5(word.encode("utf-8")).hexdigest(), 16)

class FakeDenseRetriever:
    """진짜 OpenAI 임베딩이 아니라, 같은 단어가 겹치면 벡터도 비슷해지도록 만든
    결정적(deterministic) 가짜 임베딩입니다 - 개념 체험용, 정확도는 낮습니다."""
    def __init__(self, dim=64):
        self.dim = dim
    def _embed(self, text):
        vec = [0.0] * self.dim
        for w in tokenize(text):
            h = stable_hash(w)
            for d in range(self.dim):
                vec[d] += math.sin(h * (d + 1) * 0.01)
        norm = math.sqrt(sum(v*v for v in vec)) + 1e-9
        return [v / norm for v in vec]
    def fit(self, docs):
        self.docs = docs
        self.vecs = [self._embed(d["text"]) for d in docs]
    def search(self, query, top_k=5):
        qv = self._embed(query)
        scores = [(d["id"], sum(a*b for a,b in zip(qv, v))) for d, v in zip(self.docs, self.vecs)]
        scores.sort(key=lambda x: -x[1])
        return [{"rank": r+1, "doc_id": d, "score": round(s,3)} for r,(d,s) in enumerate(scores[:top_k])]

dense = FakeDenseRetriever()
dense.fit(docs)
print("질의: '정정요청 방법'")
for r in dense.search("정정요청 방법"):
    print(r)

## 4. [융합] RRF — 실제로 우리 서비스에 반영된 바로 그 공식 ($k=60$)

In [ ]:
def rrf_fuse(sparse_hits, dense_hits, k=60, top_k=5):
    scores = {}
    for h in sparse_hits: scores[h["doc_id"]] = scores.get(h["doc_id"], 0) + 1/(k + h["rank"])
    for h in dense_hits: scores[h["doc_id"]] = scores.get(h["doc_id"], 0) + 1/(k + h["rank"])
    ranked = sorted(scores.items(), key=lambda x: -x[1])
    return [{"final_rank": r+1, "doc_id": d, "rrf_score": round(s,5)} for r,(d,s) in enumerate(ranked[:top_k])]

query = "정정요청 방법"
sp = bm25.search(query)
dn = dense.search(query)
print("🏆 RRF 최종 융합 결과:")
for r in rrf_fuse(sp, dn):
    print(r)

## 5. [질의 변환] Decomposition — 복합질문 쪼개기 (오늘 우리 서비스에 실제로 반영된 기법)

In [ ]:
def decompose(query):
    """실제 프로덕션은 LLM으로 분해하지만, 여기서는 개념 체험을 위해 규칙으로 흉내냅니다."""
    if "고" in query and ("어떻게" in query or "몇" in query or "언제" in query or "뭐" in query):
        parts = re.split(r"하고|이고|이랑", query)
        if len(parts) >= 2:
            return [p.strip() + ("?" if not p.strip().endswith("?") else "") for p in parts if p.strip()]
    return [query]

compound_q = "미술활동보고서 글자수 제한이랑 평가자 회원가입 방법이 뭐야?"
sub_qs = decompose(compound_q)
print(f"원본 복합질문: {compound_q}")
print(f"분해된 하위질문: {sub_qs}\n")

print("--- 기존(통째로 검색, top-2) : D6(회원가입)이 빠짐! ---")
for r in rrf_fuse(bm25.search(compound_q), dense.search(compound_q), top_k=2):
    print(r)

print("\n--- 분해 후 각각 검색해서 합침 : D1과 D6 둘 다 잡힘 ---")
merged = {}
for sq in sub_qs:
    for r in rrf_fuse(bm25.search(sq), dense.search(sq), top_k=1):
        merged.setdefault(r["doc_id"], r)
for r in merged.values():
    print(r)

## 6. [우리 서비스 실 사례] 오늘(2026-09-22) 홍익대학교 실제 데이터로 실측한 진짜 결과

> 아래 숫자는 위 장난감 예제가 아니라, **실제 운영 중인 art_admission 서비스**와
> **Neo4j에 저장된 진짜 홍익대학교 요강 청크**로 측정한 결과입니다.

In [ ]:
import pandas as pd

실측결과 = pd.DataFrame([
    {"기법": "가중합(기존)", "채택여부": "폐기", "실측": "9문항 평균순위 3.00"},
    {"기법": "RRF", "채택여부": "✅ 채택", "실측": "9문항 평균순위 2.67 (단, 4개 중 2개는 개별적으로 악화됨)"},
    {"기법": "HyDE", "채택여부": "❌ 기각", "실측": "9문항 평균순위 2.89 → 4.67 (악화)"},
    {"기법": "Multi-Query", "채택여부": "❌ 기각", "실측": "9문항 평균순위 2.89 → 4.00 (악화)"},
    {"기법": "Self-Query", "채택여부": "❌ 기각", "실측": "이미 배포된 RRF 하이브리드 대비 추가이득 없음"},
    {"기법": "Step-back", "채택여부": "❌ 기각", "실측": "4문항 평균순위 5.0 → 7.0 (악화)"},
    {"기법": "Decomposition", "채택여부": "✅ 채택", "실측": "복합질문 정답포함률 0/4 → 4/4 (개선)"},
])
실측결과

## ✅ 완료 판정 (Done Definition)

다음 3가지를 스스로 확인할 수 있으면 이 노트북을 이해한 것입니다:

1. **셀 4번(RRF)**을 실행해서, BM25 단독/벡터 단독 결과와 RRF 융합 결과가 서로 다를 수 있음을 확인했다.
2. **셀 5번(Decomposition)**을 실행해서, 복합질문을 쪼개면 "기존" 방식이 놓치던 D6(평가자 회원가입)이 "분해 후"엔 잡히는 걸 직접 눈으로 봤다.
3. **셀 6번(우리 서비스 실 사례)**을 보고, "이론상 좋아 보이는 기법(Multi-Query, HyDE)이 왜 실제로는 기각됐는지" 한 문장으로 설명할 수 있다.